In [36]:
import pandas as pd
import numpy as np

In [37]:
#import data from /Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/ProcessedObservedData.csv
data = pd.read_csv(r'C:\Users\jaskew\Documents\project_repository\data\processed\ProcessedObservedData.csv')
#filter by opDiv CDC
data = data[data['OpDiv'] == 'CDC']
data.drop(columns=['curr_date', 'OpDiv'], inplace=True)
# Rename the index to 'date' since 'obs_date' is now the index
data.rename(columns={'obs_date': 'date'}, inplace=True)
data['date'] = pd.to_datetime(data['date'])
data.reset_index(drop=True, inplace=True)

data.tail(10)

,indicator,API_UserName,date,observations
3496,196.251.87.59,00818860012482918321,2025-06-02,1
3497,93.123.109.230,00818860012482918321,2025-06-02,1
3498,104.21.48.1,00818860012482918321,2025-06-02,265
3499,196.251.71.232,00818860012482918321,2025-06-02,2
3500,34.160.111.145,00818860012482918321,2025-06-02,4
3501,31.177.80.32,00818860012482918321,2025-06-02,6
3502,31.177.76.32,00818860012482918321,2025-06-02,4
3503,www.shorturl.at/,00818860012482918321,2025-06-02,2
3504,104.21.54.132,00818860012482918321,2025-06-02,1
3505,104.21.76.17,00818860012482918321,2025-06-02,1


In [38]:
data[data['date'] == '2025-05-30']

,indicator,API_UserName,date,observations
3452,104.21.48.1,00818860012482918321,2025-05-30,333
3453,155.138.246.122,00818860012482918321,2025-05-30,3
3454,172.240.108.68,00818860012482918321,2025-05-30,1
3455,185.230.63.171,00818860012482918321,2025-05-30,6
3456,196.251.87.59,00818860012482918321,2025-05-30,1
3457,198.251.81.14,00818860012482918321,2025-05-30,2
3458,23.205.105.180,00818860012482918321,2025-05-30,3
3459,31.177.76.32,00818860012482918321,2025-05-30,4
3460,31.177.80.32,00818860012482918321,2025-05-30,2
3461,34.160.111.145,00818860012482918321,2025-05-30,8


In [39]:
data['date'] = pd.to_datetime(data['date'])

# Define ranges for combinations
all_users = data['API_UserName'].unique()  # Unique API_UserName
all_indicators = data['indicator'].unique()  # Unique indicators
all_dates = pd.date_range(start='2024-01-01', end=pd.Timestamp.now().normalize(), freq='D')

# Create all combinations
all_combinations = pd.MultiIndex.from_product(
    [all_users, all_dates, all_indicators],
    names=['API_UserName', 'date', 'indicator']
).to_frame(index=False)

# Merge with the existing data
merged = all_combinations.merge(data, how='left', on=['API_UserName', 'date', 'indicator'])

# Fill missing values in 'observations' with 0 (not seen)
merged['observations'] = merged['observations'].fillna(0).astype(int)

# Display the first few rows of the merged dataset

# Convert the 'date' column to datetime format
merged['date'] = pd.to_datetime(merged['date'])

# Extract day of the week (0=Monday, 6=Sunday)
merged['dayofweek'] = merged['date'].dt.dayofweek

# Determine if the day is a weekend (Saturday=5, Sunday=6)
merged['is_weekend'] = merged['dayofweek'].isin([5, 6])

# Extract additional features if needed
merged['day'] = merged['date'].dt.day
merged['month'] = merged['date'].dt.month

merged['seen'] = (merged['observations'] > 0).astype(int)
merged.head(10)
merged.to_csv(r'C:\Users\jaskew\Documents\project_repository\notebooks\observationEventForecasting\DataPreprocessing\FullIndicatorMatrix.csv', index=False)

In [40]:
merged[merged['date'] == '2025-05-30']

,API_UserName,date,indicator,observations,dayofweek,is_weekend,day,month,seen
133385,00818860012482918321,2025-05-30,146.71.50.198,0,4,False,30,5,0
133386,00818860012482918321,2025-05-30,149.36.49.225,0,4,False,30,5,0
133387,00818860012482918321,2025-05-30,162.142.125.242,0,4,False,30,5,0
133388,00818860012482918321,2025-05-30,162.142.125.247,0,4,False,30,5,0
133389,00818860012482918321,2025-05-30,162.142.125.255,0,4,False,30,5,0
...,...,...,...,...,...,...,...,...,...
133639,00818860012482918321,2025-05-30,31.177.76.32,4,4,False,30,5,1
133640,00818860012482918321,2025-05-30,31.177.80.32,2,4,False,30,5,1
133641,00818860012482918321,2025-05-30,70.39.75.187,2,4,False,30,5,1
133642,00818860012482918321,2025-05-30,190.92.174.45,0,4,False,30,5,0
